# Этап 2: baseline RandomForest (автономный)

Setup + dataset + RF + графики MAE.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Lamblador/IR_expert_system_3.git"  # при необходимости замените
REPO_DIR = Path("IR_expert_system_3")

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

%cd IR_expert_system_3
!pip install -q -e ".[torch]"
!ir-pipeline --help
import subprocess
help_txt = subprocess.check_output(['ir-pipeline', '--help'], text=True)
if ' run ' not in help_txt:
    print('WARNING: команда `run` отсутствует. Ноутбук использует fallback без run-stage.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
IR_DATA = Path('/content/drive/MyDrive/ir_data')
print('IR_DATA exists:', IR_DATA.exists(), IR_DATA)


## Датасет: загрузка вручную

1. **Files → Upload** в Colab: `dataset_v001.zip` / `dataset_mini.zip` в `/content` (или положите архив на Google Drive).
2. Выполните ячейку распаковки ниже — ожидается `data/processed/<версия>/spectra.npz`.
3. Если архива нет — следующая ячейка скачает мини-датасет с Hugging Face.


In [ ]:
from pathlib import Path
import zipfile

DATASET_VERSIONS = ('dataset_mini', 'dataset_v001')
SEARCH_ROOTS = [
    Path('/content'),
    Path('/content/IR_expert_system_3'),
    Path('/content/drive/MyDrive'),
    Path('/content/drive/MyDrive/ir_data'),
    Path('.'),
]
try:
    SEARCH_ROOTS.insert(0, IR_DATA)
except NameError:
    pass
DEST = Path('data/processed')
DEST.mkdir(parents=True, exist_ok=True)

def _dataset_ready(name: str) -> bool:
    return (DEST / name / 'spectra.npz').is_file()

def _find_zip_archives() -> list[Path]:
    found: list[Path] = []
    seen: set[str] = set()
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for p in root.rglob('*.zip'):
            key = str(p.resolve())
            if key in seen:
                continue
            low = p.name.lower()
            if any(v in low for v in DATASET_VERSIONS):
                seen.add(key)
                found.append(p)
    return sorted(found, key=lambda x: x.stat().st_mtime, reverse=True)

archives = _find_zip_archives()
print('Найденные zip с датасетом:')
if archives:
    for p in archives[:15]:
        print(f'  {p} ({p.stat().st_size / 1e6:.1f} MB)')
else:
    print('  (нет — загрузите через Files → Upload)')

for version in DATASET_VERSIONS:
    if _dataset_ready(version):
        print(f'OK: {DEST / version} уже распакован')
        continue
    matched = [p for p in archives if version in p.name.lower()]
    if not matched:
        print(f'Пропуск {version}: zip не найден')
        continue
    zp = matched[0]
    print(f'Распаковка {zp.name} → {DEST}')
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(DEST)
    if _dataset_ready(version):
        print(f'  → готово: {DEST / version / "spectra.npz"}')
    else:
        print(
            f'  WARNING: после распаковки нет {DEST / version / "spectra.npz"}. '
            'Проверьте структуру zip (внутри должна быть папка {version}/).'
        )


In [ ]:
from pathlib import Path

DATASET_DIR = Path('data/processed/dataset_mini')
if DATASET_DIR.joinpath('spectra.npz').is_file():
    print(f'OK: {DATASET_DIR}')
else:
    print('dataset_mini not found → fetching from HF...')
    !ir-pipeline fetch-data --filename dataset_mini.zip --extract-to data/processed


In [ ]:
from pathlib import Path
import zipfile, shutil

SEARCH_ROOTS = [Path('/content'), Path('/content/IR_expert_system_3'), Path('/content/drive/MyDrive')]
candidates = []
for root in SEARCH_ROOTS:
    if not root.exists():
        continue
    for p in root.rglob('*'):
        name = p.name.lower()
        if p.is_dir() and name == 'downloaded_jcamp':
            candidates.append(('dir', p))
        if p.is_file() and ('downloaded_jcamp' in name and name.endswith('.zip')):
            candidates.append(('zip', p))

print('Found candidates:')
for k, p in candidates[:30]:
    print(k, p)

target = Path('/content/IR_expert_system_3/downloaded_jcamp')
target.parent.mkdir(parents=True, exist_ok=True)
if not target.exists():
    for kind, p in candidates:
        if kind == 'dir':
            print('Copying directory to', target)
            shutil.copytree(p, target, dirs_exist_ok=True)
            break
        if kind == 'zip':
            print('Extracting zip to', target)
            target.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(p) as zf:
                zf.extractall(target)
            break
print('downloaded_jcamp exists:', target.exists())


In [ ]:
from pathlib import Path
from IPython.display import Image, display

RUN_DIR = Path('runs/colab_pipeline_rf/rf_run')
!ir-pipeline train --paths configs/paths.huggingface.yaml --dataset-version dataset_mini --mode spectrum_structure --config configs/train_mini.yaml --run-dir {RUN_DIR}
!ir-pipeline plot-train-metrics --run-dir {RUN_DIR}

for pat in ['metrics_per_band_mae.png', 'metrics_by_group_mae.png']:
    hits = sorted(Path('runs').rglob(pat))
    if hits:
        display(Image(filename=str(hits[-1]), width=900))


## Сохранить обученную модель на локальный ПК

Выполните ячейку ниже — браузер скачает zip каталога run (`models.joblib`, `metrics.json`, `irresnet_bundle.pt` и т.д.). На Windows распакуйте в `runs/<имя>/` и укажите `--run-dir`.


In [ ]:
from pathlib import Path
import shutil
from google.colab import files

RUN_DIR = Path('runs/colab_pipeline_rf/rf_run')
if not RUN_DIR.is_dir():
    raise FileNotFoundError(
        f'Нет {RUN_DIR} — сначала выполните ячейку обучения.'
    )

artifacts = [p for p in RUN_DIR.iterdir() if p.is_file()]
if not artifacts:
    raise FileNotFoundError(f'{RUN_DIR} пуст — нечего архивировать.')
print('Файлы:', [p.name for p in sorted(artifacts)])

zip_path = Path('/content/rf_run_colab.zip')
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', RUN_DIR)
size_mb = zip_path.stat().st_size / 1e6
print(f'Архив: {zip_path} ({size_mb:.2f} MB)')
files.download(str(zip_path))
print('Скачивание запущено.')
